# Mandelbrot boundary-density notebook

This notebook is the slower companion to `cpp/mandelbrot_boundary_density_svg.cpp` and `art/mandelbrot-boundary-density.csv`. The point is not to re-explain the Mandelbrot set from zero. The point is narrower: make the boundary-density card readable instead of leaving it as a nice-looking histogram.


## Question and scope

For the three views already used by the repo — the whole set, Seahorse Valley, and a mini-brot neighborhood — where does the escaping mass actually live?

Scope boundary:

- We are looking only at the sampled pixels and escape-time statistics already generated by the C++ program.
- We are **not** estimating fractal dimension, harmonic measure, or any invariant boundary law.
- The result is a practical reading aid for this artifact, not a theorem about the set.


## Escape-time reminder

For `f_c(z) = z^2 + c`, the usual sampled orbit starts at `z_0 = 0` and iterates

`z_{n+1} = z_n^2 + c`.

If `|z_n|` exceeds the bailout radius, the point is counted as escaping. A smoothed escape count is then used so the histogram is not trapped on hard integer bands.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

csv_path = Path('../art/mandelbrot-boundary-density.csv')
rows = list(csv.DictReader(csv_path.open()))
len(rows), rows[0].keys()


In [ ]:
by_view = defaultdict(list)
for row in rows:
    by_view[row['view']].append(row)

summary = {}
for view, items in by_view.items():
    total = sum(int(item['count']) for item in items)
    tail = sum(int(item['count']) for item in items if float(item['bucket_start']) >= 62.222)
    summary[view] = {
        'pixels_in_histogram': total,
        'tail_fraction_ge_62.222': tail / total,
        'escape_fraction': float(items[0]['escape_fraction']),
        'mean_smooth_iter': float(items[0]['mean_smooth_iter']),
        'p90_smooth_iter': float(items[0]['p90_smooth_iter']),
        'max_smooth_iter': float(items[0]['max_smooth_iter']),
    }
summary


## Reading the three views

A useful compact comparison is the slow-tail fraction above about 62 smoothed iterations:

- **Whole set:** only about 1.1% of the escaping mass lands in that tail. Most of the sampled exterior gets away quickly.
- **Seahorse Valley:** about 19.5% lands in the slow tail. This is already a boundary-rich view.
- **Mini-brot neighborhood:** about 30.8% lands in the slow tail. The histogram is visibly heavier because the crop keeps more points near a locally complicated boundary.

That is the whole card in one sentence: zooming inward does not just raise the average iteration count. It changes how much of the sampled image budget is spent on stubborn near-boundary points.


In [ ]:
for view, stats in summary.items():
    print(view)
    print('  escape fraction      ', f"{stats['escape_fraction']:.3f}")
    print('  mean smooth iter     ', f"{stats['mean_smooth_iter']:.3f}")
    print('  p90 smooth iter      ', f"{stats['p90_smooth_iter']:.3f}")
    print('  tail fraction >=62.2 ', f"{100*stats['tail_fraction_ge_62.222']:.2f}%")
    print()


## Adversarial check

The card is easy to over-read, so here is the short pushback:

- the histogram depends on the chosen crop
- it also depends on the resolution, bailout rule, and iteration cap
- the slow tail is a sampling statement, not a proof that one region is intrinsically 'more fractal' in any rigorous global sense

So the right reading is local and operational: for these three generated views, the deeper crops devote much more of their sampled mass to slow escape behavior.


## Small problems worth trying

1. Rebuild the card at a higher iteration cap and see which ranking is stable.
2. Replace the fixed threshold `62.222` with the view's own 90th percentile and compare normalized tail mass instead.
3. Add one more crop that is visually busy but not mini-brot-heavy, then check whether the histogram stays this tail-heavy.
4. Compare the histogram before and after doubling resolution to see how much of the story is a coarse-grid artifact.


## References and caveats

Useful background if you want the real theory instead of this artifact-level reading:

- escape-time rendering for the Mandelbrot set
- smoothed iteration counts for coloring and band reduction
- the distinction between sampled escape statistics and geometric results like dimension or measure

This notebook stays on the artifact side of that line on purpose.
